[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-9_executed.ipynb)

---


# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 9: QAOA and the Limits of Exact Classical Simulation

### Purpose and scope

This notebook formulates a capacitated facility location problem as a quadratic unconstrained binary optimization (QUBO) model and describes how the Quantum Approximate Optimization Algorithm (QAOA) acts on that model. The example has six candidate facilities and eight customer zones. Facility-opening and customer-assignment decisions are binary; capacity constraints are represented using binary slack variables.

The notebook includes an exact state-vector simulator for small QUBOs. A state vector for \(n\) qubits has \(2^n\) complex amplitudes, so the example's full QUBO is too large for this simulator. The notebook therefore builds the model and reports the resource limitation instead of claiming a QAOA result. A classical state-vector simulation is not quantum hardware and cannot establish quantum advantage.

### Model assumptions

- Each customer is assigned to exactly one facility, and each facility either serves all of an assigned customer's demand or none of it.
- A facility can serve customers only when it is open. Its total assigned demand cannot exceed its capacity.
- Transportation cost is demand multiplied by Euclidean distance and a rate of \$8.50 per coordinate-distance unit. Facility fixed costs and transportation costs are expressed in dollars.
- Demands and capacities are integer-valued, allowing capacity slack to be encoded in binary. The model does not include uncertainty, multiple products, partial shipments, or time periods.
- Penalty weights define the relative cost of constraint violations in the QUBO. They are modeling parameters, not proof that every penalty is large enough to enforce feasibility in an unconstrained QUBO minimum.

### QUBO formulation

Let \(I\) be the set of facilities and \(J\) the set of customers. Let \(x_i\in\{0,1\}\) indicate whether facility \(i\) is open, and let \(y_{ij}\in\{0,1\}\) indicate whether customer \(j\) is assigned to facility \(i\). For each facility, let \(s_i\) be a nonnegative integer slack variable encoded with binary bits. The original cost is

$$\sum_{i\in I} f_i x_i + \sum_{i\in I}\sum_{j\in J} c_{ij}d_jy_{ij},$$

where \(f_i\) is the fixed opening cost, \(c_{ij}\) is the transportation cost per demand unit, and \(d_j\) is customer demand. The constraints are encoded as squared penalties or, for the logical implication, as a binary product:

$$\begin{aligned}
E(x,y,s) ={}& \sum_{i\in I} f_i x_i + \sum_{i\in I}\sum_{j\in J} c_{ij}d_jy_{ij} \\
&+ A\sum_{j\in J}\left(\sum_{i\in I}y_{ij}-1\right)^2 \\
&+ B\sum_{i\in I}\left(\sum_{j\in J}d_jy_{ij}+s_i-C_ix_i\right)^2 \\
&+ L\sum_{i\in I}\sum_{j\in J}y_{ij}(1-x_i).
\end{aligned}$$

Here, \(C_i\) is facility capacity; \(A\), \(B\), and \(L\) are the assignment, capacity, and service-level penalty weights. The assignment square is zero precisely when each customer has one assignment. The capacity equality uses unused capacity \(s_i=C_ix_i-\sum_jd_jy_{ij}\); its binary encoding permits the required integer slack values. The service-level term penalizes an assignment to a closed facility. Expanding these expressions and using \(z^2=z\) for binary variables gives a QUBO objective \(z^TQz+\ell^Tz+q_0\), where \(z\) contains all decision and slack bits. The code stores off-diagonal coefficients symmetrically in \(Q\), so both matrix entries together contribute the intended pairwise coefficient.

### QAOA in this notebook

QAOA alternates a problem unitary, which applies the phase \(e^{-i\gamma E(z)}\) to each computational-basis state \(z\), with a mixer unitary built from single-qubit \(X\)-rotations. At depth \(p\), the circuit uses parameters \((\gamma_1,\beta_1,\ldots,\gamma_p,\beta_p)\). A classical optimizer tunes these parameters to reduce the expected QUBO energy; measurement then samples candidate bit strings. This hybrid procedure does not guarantee an optimum.

The exact simulator below enumerates the full state vector and is therefore exponential in the number of qubits. The instance is retained at its stated size; the notebook does not replace it with a smaller instance or infer a quantum speedup from a classical simulation. The optimizer and simulator are provided so that their operations and resource requirements can be studied on appropriately small QUBOs.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("CFLP QUBO and QAOA demonstration")
print("The state-vector simulator is a classical, exponential-time simulation.")
print("This notebook does not claim a quantum-computing speedup.")

In [ ]:
@dataclass
class Facility:
    """A candidate facility with location, fixed cost, and capacity."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: int


@dataclass
class Customer:
    """A customer zone with location and integer demand."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    demand: int


@dataclass
class QUBOProblem:
    """A QUBO objective and the indices of its binary variables."""

    n_facilities: int
    n_customers: int
    Q_matrix: np.ndarray
    linear_terms: np.ndarray
    constant_term: float
    variable_names: List[str]

    @property
    def n_variables(self) -> int:
        """Return the total number of binary decision and slack variables."""
        return len(self.linear_terms)

In [ ]:
def initialize_quantum_facilities() -> List[Facility]:
    """Create the candidate facilities for the example instance."""
    return [
        Facility(1, "North Hub", 20, 80, 180000, 1200),
        Facility(2, "Central DC", 50, 50, 140000, 1000),
        Facility(3, "South DC", 80, 20, 120000, 800),
        Facility(4, "East Facility", 85, 65, 160000, 900),
        Facility(5, "West Facility", 15, 35, 100000, 700),
        Facility(6, "Northeast DC", 70, 85, 150000, 1100),
    ]


def initialize_quantum_customers() -> List[Customer]:
    """Create the customer zones for the example instance."""
    return [
        Customer(1, "Store A", 25, 75, 400),
        Customer(2, "Store B", 45, 65, 350),
        Customer(3, "Store C", 75, 35, 300),
        Customer(4, "Store D", 90, 60, 320),
        Customer(5, "Store E", 30, 20, 280),
        Customer(6, "Store F", 60, 90, 380),
        Customer(7, "Store G", 85, 40, 250),
        Customer(8, "Store H", 55, 75, 360),
    ]


facilities = initialize_quantum_facilities()
customers = initialize_quantum_customers()

transport_costs = {}
for facility in facilities:
    for customer in customers:
        distance = np.hypot(
            facility.x_coord - customer.x_coord,
            facility.y_coord - customer.y_coord,
        )
        transport_costs[(facility.id, customer.id)] = distance * 8.5

print("CFLP instance initialized:")
print(f"Candidate facilities: {len(facilities)}")
print(f"Customer zones: {len(customers)}")
print(f"Facility-customer transportation costs: {len(transport_costs)}")
print("Each transportation cost is stated per demand unit.")

In [ ]:
class QUBOConverter:
    """Expand the CFLP objective and constraints into a binary QUBO."""

    def __init__(self, facilities: List[Facility], customers: List[Customer]):
        self.facilities = facilities
        self.customers = customers
        self.n_facilities = len(facilities)
        self.n_customers = len(customers)
        self.transport_costs = self._calculate_transport_costs()

    def _calculate_transport_costs(self) -> Dict[Tuple[int, int], float]:
        """Calculate Euclidean transportation cost per demand unit."""
        costs = {}
        for facility in self.facilities:
            for customer in self.customers:
                distance = np.hypot(
                    facility.x_coord - customer.x_coord,
                    facility.y_coord - customer.y_coord,
                )
                costs[(facility.id, customer.id)] = distance * 8.5
        return costs

    def convert_to_qubo(
        self, penalty_weights: Optional[Dict[str, float]] = None
    ) -> QUBOProblem:
        """Build the QUBO, including binary capacity-slack variables."""
        if penalty_weights is None:
            penalty_weights = {
                "fixed_cost": 1.0,
                "transport_cost": 1.0,
                "capacity": 1000.0,
                "assignment": 1000.0,
                "service_level": 1000.0,
            }

        required_weights = {
            "fixed_cost",
            "transport_cost",
            "capacity",
            "assignment",
            "service_level",
        }
        if set(penalty_weights) != required_weights:
            raise ValueError("Provide exactly the five documented QUBO weights.")
        if any(weight < 0 for weight in penalty_weights.values()):
            raise ValueError("QUBO weights must be nonnegative.")

        variable_names = []
        x_index = {}
        y_index = {}
        slack_indices = {}

        for i, facility in enumerate(self.facilities):
            x_index[i] = len(variable_names)
            variable_names.append(f"x_{facility.id}")

        for i, facility in enumerate(self.facilities):
            for j, customer in enumerate(self.customers):
                y_index[(i, j)] = len(variable_names)
                variable_names.append(f"y_{facility.id}_{customer.id}")

        for i, facility in enumerate(self.facilities):
            number_of_bits = int(np.ceil(np.log2(facility.capacity + 1)))
            slack_indices[i] = []
            for bit in range(number_of_bits):
                slack_indices[i].append(len(variable_names))
                variable_names.append(f"s_{facility.id}_{bit}")

        n_variables = len(variable_names)
        linear = np.zeros(n_variables, dtype=float)
        pair_terms = {}
        constant = 0.0

        def add_pair(first: int, second: int, coefficient: float) -> None:
            """Accumulate a binary pair term before symmetric matrix storage."""
            key = tuple(sorted((first, second)))
            pair_terms[key] = pair_terms.get(key, 0.0) + coefficient

        def add_square(
            coefficients: Dict[int, float], offset: float, weight: float
        ) -> None:
            """Expand weight times a squared affine expression for binary bits."""
            nonlocal constant
            constant += weight * offset**2
            items = list(coefficients.items())
            for variable, coefficient in items:
                linear[variable] += weight * (
                    coefficient**2 + 2 * offset * coefficient
                )
            for position, (first, first_coefficient) in enumerate(items):
                for second, second_coefficient in items[position + 1 :]:
                    add_pair(
                        first,
                        second,
                        2 * weight * first_coefficient * second_coefficient,
                    )

        for i, facility in enumerate(self.facilities):
            linear[x_index[i]] += penalty_weights["fixed_cost"] * facility.fixed_cost

        for i, facility in enumerate(self.facilities):
            for j, customer in enumerate(self.customers):
                linear[y_index[(i, j)]] += (
                    penalty_weights["transport_cost"]
                    * self.transport_costs[(facility.id, customer.id)]
                    * customer.demand
                )

        for j in range(self.n_customers):
            assignment = {y_index[(i, j)]: 1.0 for i in range(self.n_facilities)}
            add_square(assignment, -1.0, penalty_weights["assignment"])

        for i, facility in enumerate(self.facilities):
            capacity_expression = {x_index[i]: -float(facility.capacity)}
            for j, customer in enumerate(self.customers):
                capacity_expression[y_index[(i, j)]] = float(customer.demand)
            for bit, variable in enumerate(slack_indices[i]):
                capacity_expression[variable] = float(2**bit)
            add_square(
                capacity_expression,
                0.0,
                penalty_weights["capacity"],
            )

        for i in range(self.n_facilities):
            for j in range(self.n_customers):
                assignment_variable = y_index[(i, j)]
                linear[assignment_variable] += penalty_weights["service_level"]
                add_pair(
                    x_index[i],
                    assignment_variable,
                    -penalty_weights["service_level"],
                )

        Q = np.zeros((n_variables, n_variables), dtype=float)
        for (first, second), coefficient in pair_terms.items():
            if first == second:
                linear[first] += coefficient
            else:
                Q[first, second] += coefficient / 2
                Q[second, first] += coefficient / 2

        return QUBOProblem(
            n_facilities=self.n_facilities,
            n_customers=self.n_customers,
            Q_matrix=Q,
            linear_terms=linear,
            constant_term=constant,
            variable_names=variable_names,
        )


qubo_converter = QUBOConverter(facilities, customers)
qubo_problem = qubo_converter.convert_to_qubo()

print("QUBO construction complete:")
print(f"Facility-opening variables: {qubo_problem.n_facilities}")
print(f"Customer-assignment variables: {qubo_problem.n_facilities * qubo_problem.n_customers}")
print(f"Total binary variables, including capacity slack: {qubo_problem.n_variables}")
print(f"Q matrix shape: {qubo_problem.Q_matrix.shape}")
print(f"Linear coefficient count: {len(qubo_problem.linear_terms)}")

In [ ]:
class QuantumSimulator:
    """Exact state-vector simulator for small numbers of qubits."""

    def __init__(self, n_qubits: int, max_qubits: int = 20):
        if n_qubits < 1:
            raise ValueError("The simulator requires at least one qubit.")
        if n_qubits > max_qubits:
            raise ValueError(
                f"Exact state-vector simulation is limited here to {max_qubits} "
                f"qubits; this problem has {n_qubits}."
            )
        self.n_qubits = n_qubits
        self.max_qubits = max_qubits
        self.bitstrings = [
            format(index, f"0{n_qubits}b") for index in range(2**n_qubits)
        ]
        self.reset_state()

    def reset_state(self) -> None:
        """Prepare the equal superposition used as the QAOA initial state."""
        dimension = 2**self.n_qubits
        self.amplitudes = np.full(
            dimension, 1 / np.sqrt(dimension), dtype=complex
        )
        self.probabilities = np.abs(self.amplitudes) ** 2

    def apply_problem_unitary(self, costs: np.ndarray, gamma: float) -> None:
        """Apply exp(-i gamma C) to every computational-basis amplitude."""
        self.amplitudes *= np.exp(-1j * gamma * costs)
        self.probabilities = np.abs(self.amplitudes) ** 2

    def apply_rotation_x(self, qubit: int, angle: float) -> None:
        """Apply the single-qubit mixer rotation exp(-i angle X / 2)."""
        if not 0 <= qubit < self.n_qubits:
            raise IndexError("Qubit index is outside the state vector.")
        mask = 1 << qubit
        cosine = np.cos(angle / 2)
        sine = -1j * np.sin(angle / 2)
        for index in range(len(self.amplitudes)):
            if index & mask == 0:
                partner = index | mask
                zero_amplitude = self.amplitudes[index]
                one_amplitude = self.amplitudes[partner]
                self.amplitudes[index] = (
                    cosine * zero_amplitude + sine * one_amplitude
                )
                self.amplitudes[partner] = (
                    sine * zero_amplitude + cosine * one_amplitude
                )
        self.probabilities = np.abs(self.amplitudes) ** 2

    def measure(self) -> Tuple[str, float]:
        """Sample a bit string according to the state probabilities."""
        probabilities = self.probabilities / self.probabilities.sum()
        index = np.random.choice(len(probabilities), p=probabilities)
        return self.bitstrings[index], float(probabilities[index])

    def get_probabilities(self) -> Dict[str, float]:
        """Return the probability associated with each basis bit string."""
        return dict(zip(self.bitstrings, self.probabilities.tolist()))


print("Exact state-vector simulator class defined.")
print("Its memory and runtime grow exponentially with the qubit count.")

In [ ]:
class QAOAOptimizer:
    """QAOA circuit and classical parameter optimization for a QUBO."""

    def __init__(self, qubo_problem: QUBOProblem, max_qubits: int = 20):
        self.qubo_problem = qubo_problem
        self.n_qubits = qubo_problem.n_variables
        self.quantum_simulator = QuantumSimulator(self.n_qubits, max_qubits)
        self.costs = self._calculate_basis_costs()

    def cost_function(self, bitstring: str) -> float:
        """Evaluate the QUBO objective for one binary assignment."""
        if len(bitstring) != self.n_qubits or set(bitstring) - {"0", "1"}:
            raise ValueError("The bit string must contain one binary bit per variable.")
        vector = np.fromiter((int(bit) for bit in bitstring), dtype=float)
        return float(
            vector @ self.qubo_problem.Q_matrix @ vector
            + self.qubo_problem.linear_terms @ vector
            + self.qubo_problem.constant_term
        )

    def _calculate_basis_costs(self) -> np.ndarray:
        """Evaluate the QUBO once for every basis state in the state vector."""
        costs = np.empty(2**self.n_qubits, dtype=float)
        for index, bitstring in enumerate(self.quantum_simulator.bitstrings):
            costs[index] = self.cost_function(bitstring)
        return costs

    def apply_mixer_unitary(self, beta: float) -> None:
        """Apply the standard product of single-qubit X-rotation mixers."""
        for qubit in range(self.n_qubits):
            self.quantum_simulator.apply_rotation_x(qubit, 2 * beta)

    def qaoa_circuit(self, params: List[float], p: int) -> np.ndarray:
        """Run p alternating problem and mixer layers and return probabilities."""
        if len(params) != 2 * p:
            raise ValueError("QAOA requires one gamma and beta per layer.")
        self.quantum_simulator.reset_state()
        for layer in range(p):
            gamma = params[2 * layer]
            beta = params[2 * layer + 1]
            self.quantum_simulator.apply_problem_unitary(self.costs, gamma)
            self.apply_mixer_unitary(beta)
        return self.quantum_simulator.probabilities.copy()

    def objective_function(self, params: List[float], p: int) -> float:
        """Return the expected QUBO energy of the parameterized state."""
        probabilities = self.qaoa_circuit(params, p)
        return float(probabilities @ self.costs)

    def optimize(self, p: int = 3, max_iter: int = 100) -> Dict[str, object]:
        """Tune QAOA parameters and report the most probable measured-state candidate."""
        if p < 1 or max_iter < 1:
            raise ValueError("The layer count and iteration limit must be positive.")
        initial_params = [np.pi / 4] * (2 * p)
        bounds = [(0.0, 2 * np.pi)] * (2 * p)
        convergence_history = []

        result = minimize(
            self.objective_function,
            initial_params,
            args=(p,),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": max_iter},
            callback=lambda params: convergence_history.append(
                self.objective_function(params, p)
            ),
        )

        probabilities = self.qaoa_circuit(result.x.tolist(), p)
        best_index = int(np.argmax(probabilities))
        best_solution = self.quantum_simulator.bitstrings[best_index]
        return {
            "best_solution": best_solution,
            "best_value": self.cost_function(best_solution),
            "parameters": result.x.tolist(),
            "expected_value": float(probabilities @ self.costs),
            "convergence_history": convergence_history,
            "probabilities": probabilities,
            "optimizer_success": bool(result.success),
            "optimizer_message": str(result.message),
        }


print("QAOA circuit and parameter-optimization classes defined.")
print(f"This instance requires {qubo_problem.n_variables} binary variables.")

In [ ]:
MAX_EXACT_STATEVECTOR_QUBITS = 20
results = {}
best_result = None
best_p = None
qaoa_optimizer = None

print("QAOA state-vector feasibility check")
print(f"QUBO variables / required qubits: {qubo_problem.n_variables}")
print(f"State-vector basis states required: 2^{qubo_problem.n_variables}")
print(f"Configured exact-simulation limit: {MAX_EXACT_STATEVECTOR_QUBITS} qubits")

if qubo_problem.n_variables <= MAX_EXACT_STATEVECTOR_QUBITS:
    qaoa_optimizer = QAOAOptimizer(
        qubo_problem,
        max_qubits=MAX_EXACT_STATEVECTOR_QUBITS,
    )
    for p_value in (1, 2, 3):
        print(f"Optimizing QAOA parameters with p={p_value} layers...")
        results[p_value] = qaoa_optimizer.optimize(p=p_value, max_iter=50)
    best_p = min(results, key=lambda depth: results[depth]["expected_value"])
    best_result = results[best_p]
    print(f"Selected depth by expected QUBO energy: p={best_p}")
    print(f"Most probable bit string at that depth: {best_result['best_solution']}")
    print(f"Its QUBO energy: {best_result['best_value']:.2f}")
    print(f"State expected QUBO energy: {best_result['expected_value']:.2f}")
else:
    print("Exact simulation and parameter optimization were not run.")
    print("The complete state vector is exponentially large at this problem size.")
    print("No QAOA solution or performance comparison is inferred.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("CFLP QUBO and QAOA: Model and Simulation Scope", fontsize=15)

ax = axes[0, 0]
for facility in facilities:
    ax.scatter(
        facility.x_coord,
        facility.y_coord,
        marker="s",
        s=90,
        color="tab:blue",
        edgecolors="black",
    )
    ax.annotate(facility.name, (facility.x_coord, facility.y_coord), xytext=(4, 4), textcoords="offset points", fontsize=7)
for customer in customers:
    ax.scatter(
        customer.x_coord,
        customer.y_coord,
        marker="^",
        s=55,
        color="tab:orange",
        edgecolors="black",
    )
    ax.annotate(customer.name, (customer.x_coord, customer.y_coord), xytext=(4, 4), textcoords="offset points", fontsize=7)
ax.scatter([], [], marker="s", color="tab:blue", label="Candidate facility")
ax.scatter([], [], marker="^", color="tab:orange", label="Customer zone")
ax.set_title("Locations (coordinate units)")
ax.set_xlabel("X coordinate")
ax.set_ylabel("Y coordinate")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
cost_matrix = np.array(
    [[transport_costs[(facility.id, customer.id)] for customer in customers] for facility in facilities]
)
sns.heatmap(
    cost_matrix,
    ax=ax,
    cmap="viridis",
    xticklabels=[customer.name for customer in customers],
    yticklabels=[facility.name for facility in facilities],
    cbar_kws={"label": "Cost per demand unit ($)"},
)
ax.set_title("Transportation cost per demand unit")
ax.set_xlabel("Customer zone")
ax.set_ylabel("Candidate facility")
ax.tick_params(axis="x", rotation=45)

ax = axes[0, 2]
ax.bar([facility.name for facility in facilities], [facility.capacity for facility in facilities], color="tab:green")
ax.set_title("Facility capacities")
ax.set_xlabel("Candidate facility")
ax.set_ylabel("Demand units")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 0]
assignment_count = qubo_problem.n_facilities * qubo_problem.n_customers
slack_count = qubo_problem.n_variables - qubo_problem.n_facilities - assignment_count
variable_counts = [qubo_problem.n_facilities, assignment_count, slack_count]
ax.bar(["Open", "Assignment", "Capacity slack"], variable_counts, color=["tab:blue", "tab:orange", "tab:green"])
ax.set_title("QUBO binary-variable composition")
ax.set_ylabel("Number of binary variables")
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 1]
if results:
    depths = sorted(results)
    expected_energies = [results[depth]["expected_value"] for depth in depths]
    ax.plot(depths, expected_energies, marker="o", label="Expected QUBO energy")
    ax.set_xlabel("QAOA depth p")
    ax.set_ylabel("Expected QUBO energy")
    ax.legend()
else:
    ax.text(
        0.5,
        0.5,
        "No state-vector run: problem exceeds\nthe configured exact-simulation limit",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )
    ax.set_xlabel("QAOA depth p")
    ax.set_ylabel("Expected QUBO energy")
ax.set_title("Parameter-optimization results")
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
ax.axis("off")
ax.text(
    0.02,
    0.92,
    "Exact state-vector size grows as 2^n amplitudes.\n\n"
    f"This QUBO has n = {qubo_problem.n_variables} binary variables.\n\n"
    "A classical simulation at this size is not performed,\n"
    "and no quantum advantage is demonstrated.",
    va="top",
    fontsize=10,
    wrap=True,
)
ax.set_title("Computational scope")

plt.tight_layout()
plt.show()

In [ ]:
print("QAOA and CFLP interpretation")
print("=" * 60)
print(f"Facilities: {qubo_problem.n_facilities}")
print(f"Customer zones: {qubo_problem.n_customers}")
print(f"QUBO binary variables, including slack: {qubo_problem.n_variables}")
print(f"Exact state-vector basis size: 2^{qubo_problem.n_variables}")

if best_result is None:
    print("\nNo optimized bit string is available for this instance.")
    print("The model construction can be inspected, but exact state-vector QAOA is not run at this size.")
else:
    print(f"\nQAOA depth selected by expected energy: {best_p}")
    print(f"Most probable bit string: {best_result['best_solution']}")
    print(f"QUBO energy of that bit string: {best_result['best_value']:.2f}")
    print(f"Expected QUBO energy: {best_result['expected_value']:.2f}")
    print("A most-probable sample is not necessarily feasible or globally optimal.")

print("\nInterpretation notes:")
print("- The QUBO includes assignment, capacity, and open-facility penalties.")
print("- Penalty magnitudes affect the trade-off between operating cost and violations.")
print("- QAOA is a heuristic hybrid algorithm; its solution quality is not guaranteed.")
print("- This exact classical simulator has exponential state-vector memory and runtime.")
print("- No speedup, quantum advantage, or empirical solution-quality claim follows from this model alone.")

## Strengths, limitations, and interpretation

### What the formulation and algorithm provide

- The QUBO makes binary decisions, operating costs, and constraint penalties explicit in one objective.
- Capacity slack variables allow the integer capacity inequality to be represented by a squared equality penalty.
- QAOA offers a hybrid quantum-classical procedure: quantum-circuit layers prepare a probability distribution, while a classical optimizer tunes circuit parameters.

### Important limitations

- QAOA does not generally guarantee an optimal solution or a particular approximation quality. A measured bit string must still be checked against the original assignment, capacity, and facility-opening constraints.
- Penalty weights require careful interpretation. A low-penalty QUBO minimum may violate the original constraints; selecting large penalties can also change the scale and conditioning of parameter optimization.
- Exact state-vector simulation requires memory proportional to \(2^n\) amplitudes. The full example therefore exceeds the simulator limit used here.
- Superposition does not mean that all candidate solutions can be read out simultaneously. Measurement returns sampled classical bit strings, and repeated sampling and classical processing have costs.
- This notebook does not compare QAOA against a validated classical optimizer, and it makes no claim of quantum speedup or quantum advantage.

### Appropriate interpretation

The example is useful for studying the steps from a capacitated facility-location model to a QUBO and from a QUBO to QAOA circuit operations. It is not a production facility-location solution or an empirical demonstration of quantum performance. Larger instances require methods and hardware whose computational and modeling assumptions are assessed separately.